In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler, PolynomialFeatures


def add_lag_features(df):
    df = df.copy()
    df["u_in_diff"] = df.groupby("breath_id")["u_in"].diff().fillna(0)
    df["u_out_diff"] = df.groupby("breath_id")["u_out"].diff().fillna(0)
    df["u_in_lag1"] = df.groupby("breath_id")["u_in"].shift(1).fillna(0)
    df["u_out_lag1"] = df.groupby("breath_id")["u_out"].shift(1).fillna(0)
    return df


def add_cumulative_features(df):
    df = df.copy()
    df["cum_u_in"] = df.groupby("breath_id")["u_in"].cumsum()
    df["cum_u_out"] = df.groupby("breath_id")["u_out"].cumsum()
    return df


def add_breath_aggregates(df):
    df = df.copy()
    agg = df.groupby("breath_id").agg(
        breath_len=("id", "size"),
        breath_mean_u_in=("u_in", "mean"),
        breath_mean_u_out=("u_out", "mean"),
    )
    df = df.merge(agg, left_on="breath_id", right_index=True, how="left")
    return df


train_path = "../input/ventilator-pressure-prediction/train.csv"
test_path = "../input/ventilator-pressure-prediction/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Feature engineering
train_df = add_lag_features(train_df)
test_df = add_lag_features(test_df)

train_df = add_cumulative_features(train_df)
test_df = add_cumulative_features(test_df)

train_df = add_breath_aggregates(train_df)
test_df = add_breath_aggregates(test_df)

# Include breath_id directly as a numeric feature
feature_cols = [
    "breath_id",  # newly added
    "R",
    "C",
    "time_step",
    "u_in",
    "u_out",
    "u_in_diff",
    "u_out_diff",
    "u_in_lag1",
    "u_out_lag1",
    "cum_u_in",
    "cum_u_out",
    "breath_len",
    "breath_mean_u_in",
    "breath_mean_u_out",
]

X = train_df[feature_cols]
y = train_df["pressure"]

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)

# Higher‑order polynomial features (degree 3)
poly = PolynomialFeatures(degree=3, include_bias=False)
X_train = poly.fit_transform(X_train_scaled)
X_val = poly.transform(X_val_scaled)

# Ridge with lighter regularisation
model = Ridge(alpha=0.01, random_state=42)
model.fit(X_train, y_train)

val_pred = model.predict(X_val)
val_mae = mean_absolute_error(y_val, val_pred)
print(f"Validation MAE (poly deg3 + scaling + Ridge α=0.01): {val_mae:.5f}")

# Train on full data
X_full_scaled = scaler.fit_transform(X)
X_full = poly.fit_transform(X_full_scaled)
model.fit(X_full, y)

# Predict test set
X_test_scaled = scaler.transform(test_df[feature_cols])
X_test = poly.transform(X_test_scaled)
test_pred = model.predict(X_test)




ValueError: Indexing a matrix of 3984665400 elements would incur an in integer overflow in LAPACK. Try using numpy.linalg.svd instead.

In [2]:
submission = pd.DataFrame({"id": test_df["id"], "pressure": test_pred})
submission.to_csv("submission.csv", index=False)
print("Submission file written to submission.csv")

NameError: name 'test_pred' is not defined